# Import libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Get the data

In [4]:
# Load the dataset from the CSV file
dataset = pd.read_csv('../data/book700k-800k.csv')
df = pd.DataFrame({'Id': dataset['Id'],
                        'Name': dataset['Name'],
                        'Authors': dataset['Authors'],
                        'Rating': dataset['Rating'],
                        'Description': dataset['Description']})

# Display the first few rows of the dataset
df.head


<bound method NDFrame.head of            Id                                               Name  \
0      700000  A Passion to Preserve: Gay Men as Keepers of C...   
1      700002  Culture Keepers-Florida: Oral History of the A...   
2      700003  Holiday Favorites: The Best of the Williams-So...   
3      700004  Soups, Salads & Starters: the Best of Williams...   
4      700005                              Breakfasts & Brunches   
...       ...                                                ...   
54268  799991           Piano Concerto Highlights for Solo Piano   
54269  799993  Noggin King of the Nogs (The Sagas of Noggin t...   
54270  799994  No Greater Glory: The Four Immortal Chaplains ...   
54271  799996  The White Company by Arthur Conan Doyle, Ficti...   
54272  799997                    Livewire Real Lives Dawn Fraser   

                     Authors  Rating  \
0               Will Fellows    3.75   
1      Deborah Johnson-Simon    0.00   
2            Allen Rosenberg    4

In [96]:
# Replace Nan values with ''
df['Description'] = df['Description'].fillna('')
ori_description = df['Description']

In [10]:
df['Description'][0]

'From large cities to rural communities, gay men have long been impassioned pioneers as keepers of culture: rescuing and restoring decrepit buildings, revitalizing blighted neighborhoods, saving artifacts and documents of historical significance. <i>A Passion to Preserve</i> explores this authentic and complex dimension of gay men’s lives by profiling early and contemporary preservationists from throughout the United States, highlighting contributions to the larger culture that gays are exceptionally inclined to make.'

# Data preprocessing

In [ ]:
import re
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

# Initialize the lemmatizer
lemmatizer = WordNetLemmatizer()

# Initialize the stopwords
stop_words = set(stopwords.words('english'))

def preprocessing_text(text):
    # Convert the input text to string
    text = str(text)
    
    # Convert text to lowercase
    text = text.lower()
    
    # Remove special characters and digits and replace them with a spcae
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    
    # Tokenize the text
    tokens = nltk.word_tokenize(text)
    
    # Remove stop words
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatize the tokens (convert words into their base dictionary form, ex. cats->cat)
    tokens = [lemmatizer.lemmatize(word, pos='v') for word in tokens]
    
    # Return the processed text as a string
    return " ".join(tokens)


def preprocess_dataframe(df, column_name):
    df[column_name ]= df[column_name].apply(preprocessing_text)
    return df

df = preprocess_dataframe(df=df, column_name='Description')


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\thlam\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\thlam\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\thlam\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [21]:
df['Description'].head

<bound method NDFrame.head of 0        large city rural community gay men long impass...
1                                                         
2        collector edition feature fabulous full color ...
3                                                         
4        america respect cookware retailer world larges...
                               ...                        
54268    concerto every pianist pinnacle performance ra...
54269    king nog br ice dragon br fly machine br omrud...
54270    sink dorchester icy water greenland shortly mi...
54271    hilt cry cross narrow sea would find thick bee...
54272    book tell story dawn fraser vote greatest fema...
Name: Description, Length: 54273, dtype: str>

# Get text features using tf-idf

In [100]:
from sklearn.feature_extraction.text import TfidfVectorizer

def get_text_feature(df):
    description = df['Description']
    # Text feature
    tfidf = TfidfVectorizer(stop_words="english",
                            strip_accents='ascii',
                            token_pattern=r'\w+')

    tfidf_matrix = tfidf.fit_transform(description)
    tfidf.get_feature_names_out()
    
    return tfidf_matrix
tfidf_matrix = get_text_feature(df=df)

# Calculate cosine similarity
`from sklearn.metrics.pairwise import cosine_similarity`

In [60]:
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity_of_a_target_book(book, tfidf_matrix):
    """Calculate the cosine similarity between a target book and all books
    in the TF-IDF matrix.

    Args:
        book (scipy.sparse.csr_matrix): 
            A single TF-IDF row vector representing the target book.
            Shape should be (1, n_features).
        tfidf_matrix (scipy.sparse.csr_matrix_): 
            TF-IDF matrix containing all book vectors.
            Shape should be (n_books, n_features).

    Returns:
        numpy.ndarray: A 2D array containing cosine similarity scores between the target book and 
        every book in the TF-IDF matrix.
        Shape will be (1, n_books).
    """
    X = book
    Y = tfidf_matrix
    cs = cosine_similarity(X, Y)
    return cs

book = calculate_cosine_similarity_of_a_target_book(tfidf_matrix=tfidf_matrix, book=tfidf_matrix[0])
book

array([[1.        , 0.        , 0.        , ..., 0.03131061, 0.00277451,
        0.00542904]], shape=(1, 54273))

# Add indices to the similarity array

In [44]:
tfidf_matrix.shape[0]

54273

In [62]:
def add_indices(sim_array):
    indexed_array = []
    for i, s in zip(range(tfidf_matrix.shape[0]), sim_array[0]):
        indexed_array.append((i, s))
    return indexed_array
    
book_idx = add_indices(book)
book_idx

[(0, np.float64(1.0000000000000002)),
 (1, np.float64(0.0)),
 (2, np.float64(0.0)),
 (3, np.float64(0.0)),
 (4, np.float64(0.0)),
 (5, np.float64(0.004343602419170702)),
 (6, np.float64(0.0)),
 (7, np.float64(0.0)),
 (8, np.float64(0.02560800854693365)),
 (9, np.float64(0.005483288853473634)),
 (10, np.float64(0.014774125071541046)),
 (11, np.float64(0.001715126662856703)),
 (12, np.float64(0.037609786801544615)),
 (13, np.float64(0.0)),
 (14, np.float64(0.02898527390424989)),
 (15, np.float64(0.0)),
 (16, np.float64(0.01481165486550844)),
 (17, np.float64(0.0027762287508407255)),
 (18, np.float64(0.003977473154942162)),
 (19, np.float64(0.022580899970841478)),
 (20, np.float64(0.011388665056520492)),
 (21, np.float64(0.006782134025799732)),
 (22, np.float64(0.04438433388409871)),
 (23, np.float64(0.0)),
 (24, np.float64(0.028444166959296278)),
 (25, np.float64(0.02414786837589122)),
 (26, np.float64(0.01228285851199043)),
 (27, np.float64(0.0)),
 (28, np.float64(0.0)),
 (29, np.float6

# Sort descendent

In [63]:
book_idx

[(0, np.float64(1.0000000000000002)),
 (1, np.float64(0.0)),
 (2, np.float64(0.0)),
 (3, np.float64(0.0)),
 (4, np.float64(0.0)),
 (5, np.float64(0.004343602419170702)),
 (6, np.float64(0.0)),
 (7, np.float64(0.0)),
 (8, np.float64(0.02560800854693365)),
 (9, np.float64(0.005483288853473634)),
 (10, np.float64(0.014774125071541046)),
 (11, np.float64(0.001715126662856703)),
 (12, np.float64(0.037609786801544615)),
 (13, np.float64(0.0)),
 (14, np.float64(0.02898527390424989)),
 (15, np.float64(0.0)),
 (16, np.float64(0.01481165486550844)),
 (17, np.float64(0.0027762287508407255)),
 (18, np.float64(0.003977473154942162)),
 (19, np.float64(0.022580899970841478)),
 (20, np.float64(0.011388665056520492)),
 (21, np.float64(0.006782134025799732)),
 (22, np.float64(0.04438433388409871)),
 (23, np.float64(0.0)),
 (24, np.float64(0.028444166959296278)),
 (25, np.float64(0.02414786837589122)),
 (26, np.float64(0.01228285851199043)),
 (27, np.float64(0.0)),
 (28, np.float64(0.0)),
 (29, np.float6

In [93]:
def sort_des(idx_array):
    sorted_array = sorted(idx_array, key=lambda x: x[1], reverse=True)
    return sorted_array[1:11]

sorted_array = sort_des(idx_array=book_idx)
sorted_array

[(15012, np.float64(0.32975739542736043)),
 (13832, np.float64(0.32438855339360567)),
 (40495, np.float64(0.2944226623400963)),
 (23708, np.float64(0.2814570434343624)),
 (17247, np.float64(0.27820217005352105)),
 (6879, np.float64(0.27788004343571726)),
 (5935, np.float64(0.2766622924502505)),
 (16360, np.float64(0.2682139564220764)),
 (30718, np.float64(0.2630998656455231)),
 (39114, np.float64(0.2594407783148964))]

# Get the book names from the indices

In [ ]:
sorted_array[1:6]

(15012, np.float64(0.32975739542736043))

In [101]:
def get_book_name(rec_books):
    rec_books_info = []
    for i in rec_books:
        
        rec_books_info.append(df.iloc[i[0], :])
        
    return pd.DataFrame(rec_books_info)

result = get_book_name(sorted_array)
# result = pd.DataFrame(result)
result

,Id,Name,Authors,Rating,Description
15012,727604,The Soul Beneath the Skin: The Unseen Hearts a...,David Nimmons,3.88,surprise think provoke book begin obvious fact...
13832,725500,Life Outside: The Signorile Report on Gay Men:...,Michelangelo Signorile,3.68,strong popular em em magazine columnist michel...
40495,774575,Gay Men at the Millennium,Michael Lowenthal,4.40,core issue face gay community close millennium...
23708,743801,Lavender Culture,Karla Jay,3.55,influence gay lesbian language literature thea...
17247,731667,Queer Wars: The New Gay Right and Its Critics,Paul A. Robinson,4.05,rebellion stonewall recent battle sex marriage...
6879,712601,Art and Sex in Greenwich Village: A Memoir of ...,Felice Picano,3.82,decade stonewall rebellion small gay press nam...
5935,710934,Gay by the Bay: A History of Queer Culture in ...,Susan Stryker,3.89,fabulous montage word image first book ever ch...
16360,730029,"If You Seduce a Straight Person, Can You Make ...",John P. De Cecco,4.00,debate whether people bear homosexual biologic...
30718,756813,The Gay Man's Instruction Manual: Advice for a...,Joshua Michaelmas,3.00,finally sound advice gay men today world write...
39114,772020,John Gay and the London Theatre,Calhoun Winton,3.00,beggar opera often refer today first musical c...


# Put everything together

In [110]:
def get_rec_books(book_idx, df):
    data = preprocess_dataframe(df=df, column_name='Description')
    matrix = get_text_feature(data)
    book_sim = calculate_cosine_similarity_of_a_target_book(book=matrix[book_idx], tfidf_matrix=matrix)
    book_sim = add_indices(book_sim)
    book_sim = sort_des(book_sim)
    result = get_book_name(rec_books=book_sim)
    
    return result

rec_books = get_rec_books(df=df, book_idx=2)
    

In [113]:
rec_books

,Id,Name,Authors,Rating,Description
53714,799047,The Chef's Art: Secrets of Four-Star Cooking a...,Wayne Gisslen,4.44,explain cook work organize step order prepare ...
27608,751055,Pasta Sauces,Time-Life Books,3.75,volume include information home cook need equi...
21345,739339,Vegetarian Cooking: For Beginners,Fiona Watt,4.25,step step illustration tasty delicious recipe ...
21413,739490,Pasta & Pizza for Beginners,Fiona Watt,4.20,step step illustration tasty delicious recipe ...
14333,726362,A French Chef Cooks at Home,Jacques Pépin,4.11,author entitle book jacques pepin french chef ...
31213,757728,Cooking at The Merchant House,Shaun Hill,4.00,cook merchant house show excite home cook reci...
34789,764111,The Complete Guide to Feng Shui,Gill Hale,3.69,practical reference series cover everything he...
27717,751247,Betty Crocker Cookbook: Everything You Need to...,Betty Crocker,4.26,contain mix recipe classic recipe focus health...
34629,763852,The Complete Cooking Light Cookbook,Cooking Light Magazine,4.14,cook interest one stop guide make healthy deli...
7482,713646,Fast Fixes with Mixes: 355 Delicious Recipes f...,Taste of Home,4.02,unique book contain delicious recipe cook acro...


In [115]:
for book in rec_books: 
    print(f"{rec_books.iloc[:, 1]}")

53714    The Chef's Art: Secrets of Four-Star Cooking a...
27608                                         Pasta Sauces
21345                    Vegetarian Cooking: For Beginners
21413                          Pasta & Pizza for Beginners
14333                          A French Chef Cooks at Home
31213                        Cooking at The Merchant House
34789                      The Complete Guide to Feng Shui
27717    Betty Crocker Cookbook: Everything You Need to...
34629                  The Complete Cooking Light Cookbook
7482     Fast Fixes with Mixes: 355 Delicious Recipes f...
Name: Name, dtype: str
53714    The Chef's Art: Secrets of Four-Star Cooking a...
27608                                         Pasta Sauces
21345                    Vegetarian Cooking: For Beginners
21413                          Pasta & Pizza for Beginners
14333                          A French Chef Cooks at Home
31213                        Cooking at The Merchant House
34789                      The Co

In [109]:
df.iloc[2, 1]

'Holiday Favorites: The Best of the Williams-Sonoma Kitchen Library'